In [ ]:

import time
import cupy as cp
from cupyx.scipy.interpolate import RegularGridInterpolator

ModuleNotFoundError: No module named 'cupy'

In [ ]:







class FRGGrid:
    """
    GPU implementation of the running functions:

        f_nu(p, omega)
        f_d(p, omega)

    Inside the grid:
        cupyx.scipy.interpolate.RegularGridInterpolator is used.

    Outside the grid:
        Power-law extrapolation is used.
    """

    def __init__(self, p_grid, omega_grid, f_nu_grid, f_d_grid, tail_points=200, fallback_alpha_p=2.0, fallback_alpha_w=1.0):
        self.p_grid = cp.asarray(p_grid, dtype=cp.float64)
        self.omega_grid = cp.asarray(omega_grid, dtype=cp.float64)
        self.f_nu_grid = cp.asarray(f_nu_grid, dtype=cp.float64)
        self.f_d_grid = cp.asarray(f_d_grid, dtype=cp.float64)

        expected_shape = (self.p_grid.size, self.omega_grid.size)

        if self.f_nu_grid.shape != expected_shape:
            raise ValueError(f"f_nu_grid has the wrong shape. Expected {expected_shape}, received {self.f_nu_grid.shape}.")

        if self.f_d_grid.shape != expected_shape:
            raise ValueError(f"f_d_grid has the wrong shape. Expected {expected_shape}, received {self.f_d_grid.shape}.")

        if not bool(cp.all(cp.diff(self.p_grid) > 0.0).item()):
            raise ValueError("p_grid must be strictly increasing.")

        if not bool(cp.all(cp.diff(self.omega_grid) > 0.0).item()):
            raise ValueError("omega_grid must be strictly increasing.")

        self.tail_points = int(tail_points)
        self.fallback_alpha_p = float(fallback_alpha_p)
        self.fallback_alpha_w = float(fallback_alpha_w)

        self.alpha_p_nu, self.alpha_w_nu = self._fit_power_law_exponents(self.f_nu_grid)
        self.alpha_p_d, self.alpha_w_d = self._fit_power_law_exponents(self.f_d_grid)

        self.f_nu_interp = self._make_interpolator(self.f_nu_grid)
        self.f_d_interp = self._make_interpolator(self.f_d_grid)

    def _make_interpolator(self, values):
        """Construct a GPU interpolator over (p, omega)."""
        return RegularGridInterpolator((self.p_grid, self.omega_grid), values, method="linear", bounds_error=False, fill_value=None)

    def _fit_power_law_exponents(self, values):
        """
        Fit:

            |f(p, omega)| = C p^(-alpha_p) |omega|^(-alpha_w)

        using the final tail_points x tail_points block.
        """
        n_p = min(self.tail_points, self.p_grid.size)
        n_w = min(self.tail_points, self.omega_grid.size)

        p_tail = self.p_grid[-n_p:]
        omega_tail = self.omega_grid[-n_w:]
        f_tail = values[-n_p:, -n_w:]

        P, W = cp.meshgrid(p_tail, omega_tail, indexing="ij")

        abs_f = cp.abs(f_tail)
        abs_w = cp.abs(W)
        eps = 1e-14

        mask = cp.isfinite(abs_f) & cp.isfinite(P) & cp.isfinite(abs_w) & (abs_f > eps) & (P > eps) & (abs_w > eps)

        # This scalar extraction synchronizes once during construction.
        number_valid = int(cp.count_nonzero(mask).item())

        if number_valid < 10:
            return self.fallback_alpha_p, self.fallback_alpha_w

        y = cp.log(abs_f[mask])
        log_p = cp.log(P[mask])
        log_w = cp.log(abs_w[mask])

        design_matrix = cp.column_stack((cp.ones_like(y), -log_p, -log_w))
        coefficients, _, _, _ = cp.linalg.lstsq(design_matrix, y, rcond=None)

        # Store exponents as ordinary Python scalars.
        alpha_p = float(cp.clip(coefficients[1], -10.0, 10.0).item())
        alpha_w = float(cp.clip(coefficients[2], -10.0, 10.0).item())

        return alpha_p, alpha_w

    def _eval_power_law(self, interpolator, p, omega, alpha_p, alpha_w):
        """
        GPU batch-compatible interpolation and power-law extrapolation.

        Parameters
        ----------
        p : scalar, NumPy array or CuPy array
            Momentum magnitudes.

        omega : scalar, NumPy array or CuPy array
            Frequencies, broadcast-compatible with p.

        Returns
        -------
        cupy.ndarray
            Shape obtained by broadcasting p and omega. Scalar inputs
            return a zero-dimensional CuPy array.
        """
        p, omega = cp.broadcast_arrays(cp.asarray(p, dtype=cp.float64), cp.asarray(omega, dtype=cp.float64))

        scalar_output = p.ndim == 0
        output_shape = p.shape

        p_flat = p.ravel()
        omega_flat = omega.ravel()

        # Avoid cp.any(...).item() in the hot path by validating with
        # the minimum scalar once.
        if bool(cp.any(p_flat < 0.0).item()):
            raise ValueError("Momentum magnitudes cannot be negative.")

        p_clip = cp.clip(p_flat, self.p_grid[0], self.p_grid[-1])
        omega_clip = cp.clip(omega_flat, self.omega_grid[0], self.omega_grid[-1])

        interpolation_points = cp.column_stack((p_clip, omega_clip))
        boundary_values = interpolator(interpolation_points)

        eps = 1e-14

        # ------------------------------------------------------------
        # Momentum extrapolation
        # ------------------------------------------------------------

        high_p = p_flat > self.p_grid[-1]

        p_ratio = cp.where(
            high_p,
            p_flat / cp.maximum(cp.abs(p_clip), eps),
            1.0,
        )

        momentum_factor = p_ratio ** (-alpha_p)

        # ------------------------------------------------------------
        # Frequency extrapolation
        # ------------------------------------------------------------

        outside_omega = (omega_flat > self.omega_grid[-1]) | (omega_flat < self.omega_grid[0])

        omega_ratio = cp.where(
            outside_omega,
            cp.abs(omega_flat) / cp.maximum(cp.abs(omega_clip), eps),
            1.0,
        )

        frequency_factor = omega_ratio ** (-alpha_w)

        result = (boundary_values * momentum_factor * frequency_factor).reshape(output_shape)

        # Keep scalar results on the GPU and avoid result.item().
        if scalar_output:
            return result.reshape(())

        return result

    def f_nu(self, p, omega):
        """Evaluate f_nu on the GPU."""
        return self._eval_power_law(self.f_nu_interp, p, omega, self.alpha_p_nu, self.alpha_w_nu)

    def f_d(self, p, omega):
        """Evaluate f_d on the GPU."""
        return self._eval_power_law(self.f_d_interp, p, omega, self.alpha_p_d, self.alpha_w_d)



class FlowParameters:
    """
    GPU-compatible RG-flow parameters and regulator functions.

    Momentum inputs may have shape:

        (d,)
        (B, d)
        (..., d)

    Scalar momentum inputs return zero-dimensional CuPy arrays rather
    than Python scalars, avoiding GPU-to-CPU synchronization.
    """

    def __init__(self, nu0=1.0, D0=1.0, k0=1.0, eta_nu=4.0/3.0, eta_D=3.0, a=0.5):
        self.nu0 = float(nu0)
        self.D0 = float(D0)
        self.k0 = float(k0)
        self.eta_nu = float(eta_nu)
        self.eta_D = float(eta_D)
        self.a = float(a)

    @staticmethod
    def _as_vector(p):
        return cp.asarray(p, dtype=cp.float64)

    @staticmethod
    def _restore_scalar(result, scalar_output):
        return result.reshape(()) if scalar_output else result

    def _p_data(self, p):
        p = self._as_vector(p)

        if p.ndim < 1:
            raise ValueError("p must have shape (d,) or (..., d).")

        p2 = cp.sum(p * p, axis=-1)
        pmag = cp.sqrt(p2)

        return p, p2, pmag

    @staticmethod
    def _delta(i, j):
        return 1.0 if i == j else 0.0

    def nu_k(self, k_rg):
        """nu_k = nu0 * (k_rg / k0)^(-eta_nu)."""
        k_rg = float(k_rg)

        if k_rg <= 0.0:
            raise ValueError("k_rg must be positive.")

        return self.nu0 * (k_rg / self.k0)**(-self.eta_nu)

    def D_k(self, k_rg):
        """D_k = D0 * (k_rg / k0)^(-eta_D)."""
        k_rg = float(k_rg)

        if k_rg <= 0.0:
            raise ValueError("k_rg must be positive.")

        return self.D0 * (k_rg / self.k0)**(-self.eta_D)

    def n_hat(self, x):
        """n_hat(x) = exp(-x^2)."""
        x = cp.asarray(x, dtype=cp.float64)
        return cp.exp(-(x * x))

    def r_hat(self, x):
        """
        r_hat(x) = a / (exp(x) - 1).

        The value is singular at x=0.
        """
        x = cp.asarray(x, dtype=cp.float64)
        denominator = cp.expm1(x)
        return cp.where(x == 0.0, cp.inf, self.a / denominator)

    def r_hat_prime(self, x, eps=1e-14):
        """
        r_hat'(x) = -a exp(x) / (exp(x) - 1)^2.
        """
        x = cp.asarray(x, dtype=cp.float64)

        # Stable representation:
        # exp(x)/(exp(x)-1)^2 = exp(-x)/(1-exp(-x))^2
        exp_minus_x = cp.exp(-x)
        denominator = -cp.expm1(-x)
        safe_denominator = cp.where(cp.abs(denominator) > eps, denominator, 1.0)

        regular_value = -self.a * exp_minus_x / safe_denominator**2

        return cp.where(cp.abs(x) < eps, -cp.inf, regular_value)

    def R_scalar(self, p, k_rg, eps=1e-14):
        """
        Batch-compatible scalar regulator.

        Parameters
        ----------
        p : CuPy-compatible array
            Shape (d,) or (...,d).

        Returns
        -------
        cupy.ndarray
            Shape p.shape[:-1].
        """
        p = cp.asarray(p, dtype=cp.float64)
        k_rg = float(k_rg)

        if p.ndim < 1:
            raise ValueError("p must have shape (d,) or (..., d).")

        if k_rg <= 0.0:
            raise ValueError("k_rg must be positive.")

        scalar_output = p.ndim == 1
        p_sq = cp.sum(p * p, axis=-1)

        nu = self.nu_k(k_rg)
        x = p_sq / k_rg**2

        denominator = cp.expm1(x)
        safe_denominator = cp.where(cp.abs(denominator) > eps, denominator, 1.0)

        regular_value = nu * p_sq * self.a / safe_denominator
        zero_value = nu * self.a * k_rg**2

        result = cp.where(p_sq < eps, zero_value, regular_value)

        return self._restore_scalar(result, scalar_output)

    def Nkappa_scalar(self, p, k_rg):
        """
        Batch-compatible scalar forcing profile.

        Nkappa(p) = D_k (p^2/k^2) exp(-p^2/k^2).
        """
        p = cp.asarray(p, dtype=cp.float64)
        k_rg = float(k_rg)

        if p.ndim < 1:
            raise ValueError("p must have shape (d,) or (..., d).")

        if k_rg <= 0.0:
            raise ValueError("k_rg must be positive.")

        scalar_output = p.ndim == 1
        p_sq = cp.sum(p * p, axis=-1)

        D = self.D_k(k_rg)
        x_sq = p_sq / k_rg**2
        result = D * x_sq * cp.exp(-x_sq)

        return self._restore_scalar(result, scalar_output)

    def R_k(self, p, i, j, k_rg):
        """R_k,ij(p) = delta_ij R_k(p)."""
        return self._delta(i, j) * self.R_scalar(p, k_rg)

    def Nkappa_k(self, p, i, j, k_rg):
        """Nkappa_ij(p) = delta_ij Nkappa(p)."""
        return self._delta(i, j) * self.Nkappa_scalar(p, k_rg)

    def R_matrix(self, p, k_rg):
        """
        Batch-compatible regulator matrix.

        Input
        -----
        p : (d,) or (...,d)

        Output
        ------
        (d,d) or (...,d,d)
        """
        p = cp.asarray(p, dtype=cp.float64)

        if p.ndim < 1:
            raise ValueError("p must have shape (d,) or (..., d).")

        d_space = p.shape[-1]
        R = self.R_scalar(p, k_rg)

        return R[..., None, None] * cp.eye(d_space, dtype=cp.complex128)

    def Nkappa_matrix(self, p, k_rg):
        """
        Batch-compatible noise matrix.

        Input
        -----
        p : (d,) or (...,d)

        Output
        ------
        (d,d) or (...,d,d)
        """
        p = cp.asarray(p, dtype=cp.float64)

        if p.ndim < 1:
            raise ValueError("p must have shape (d,) or (..., d).")

        d_space = p.shape[-1]
        Nkappa = self.Nkappa_scalar(p, k_rg)

        return Nkappa[..., None, None] * cp.eye(d_space, dtype=cp.complex128)

    def dRds_scalar(self, p, k_rg, eps=1e-14):
        """
        Batch-compatible derivative partial_s R_k, where s=ln(k).

        partial_s R_k =
            nu_k p^2 [-eta_nu r(x) - 2x r'(x)]

        with x=p^2/k^2.
        """
        p = cp.asarray(p, dtype=cp.float64)
        k_rg = float(k_rg)

        if p.ndim < 1:
            raise ValueError("p must have shape (d,) or (..., d).")

        if k_rg <= 0.0:
            raise ValueError("k_rg must be positive.")

        scalar_output = p.ndim == 1
        p_sq = cp.sum(p * p, axis=-1)

        nu = self.nu_k(k_rg)
        x = p_sq / k_rg**2

        exp_minus_x = cp.exp(-x)
        denominator = -cp.expm1(-x)
        safe_denominator = cp.where(cp.abs(denominator) > eps, denominator, 1.0)

        r_value = self.a * exp_minus_x / safe_denominator
        r_prime = -self.a * exp_minus_x / safe_denominator**2

        regular_value = nu * p_sq * (-self.eta_nu * r_value - 2.0 * x * r_prime)
        zero_value = (2.0 - self.eta_nu) * nu * self.a * k_rg**2

        result = cp.where(p_sq < eps, zero_value, regular_value)

        return self._restore_scalar(result, scalar_output)

    def dRds_k(self, p, i, j, k_rg):
        """partial_s R_k,ij = delta_ij partial_s R_k."""
        return self._delta(i, j) * self.dRds_scalar(p, k_rg)

    def dNkappads_scalar(self, p, k_rg):
        """
        Batch-compatible partial_s Nkappa.

        Nkappa = D_k x^2 exp(-x^2), x=|p|/k.

        partial_s Nkappa =
            Nkappa (-eta_D - 2 + 2x^2).
        """
        p = cp.asarray(p, dtype=cp.float64)
        k_rg = float(k_rg)

        if p.ndim < 1:
            raise ValueError("p must have shape (d,) or (..., d).")

        if k_rg <= 0.0:
            raise ValueError("k_rg must be positive.")

        scalar_output = p.ndim == 1
        p_sq = cp.sum(p * p, axis=-1)

        D = self.D_k(k_rg)
        x_sq = p_sq / k_rg**2
        Nkappa = D * x_sq * cp.exp(-x_sq)

        result = Nkappa * (-self.eta_D - 2.0 + 2.0 * x_sq)

        return self._restore_scalar(result, scalar_output)

    def dNkappadk_scalar(self, p, k_rg):
        """partial_k Nkappa = partial_s Nkappa / k."""
        k_rg = float(k_rg)

        if k_rg <= 0.0:
            raise ValueError("k_rg must be positive.")

        return self.dNkappads_scalar(p, k_rg) / k_rg

    def dNkappads_k(self, p, i, j, k_rg):
        """partial_s Nkappa_ij = delta_ij partial_s Nkappa."""
        return self._delta(i, j) * self.dNkappads_scalar(p, k_rg)

    def dNkappadk_k(self, p, i, j, k_rg):
        """partial_k Nkappa_ij = delta_ij partial_k Nkappa."""
        return self._delta(i, j) * self.dNkappadk_scalar(p, k_rg)


        """Batch-compatible D_d."""
        scalar_output = self._check_scalar(p,omega1)  # since, right from the _rhs_prohected function....only bactched momenta arguments including the external momenta are passed
        # one omega should be sufficient
        p_batch = p
        omega1_batch, omega2_batch, omega3_batch = omega1,omega2,omega3
        p_mag = np.linalg.norm(p_batch, axis=-1)
        omega12 = omega1_batch + omega2_batch
        omega123 = omega12 + omega3_batch

        term1 = self._safe_frequency_square_ratio(self.grid.f_d(p_mag, omega123), omega123)
        term2 = self._safe_frequency_square_ratio(self.grid.f_d(p_mag, omega12), omega12)
        term3 = self._safe_frequency_square_ratio(self.grid.f_d(p_mag, omega1_batch), omega1_batch)
        result = term1 + term2 + term3

        return self._restore_scalar_output(result, scalar_output)

class Vertices:
    def __init__(self, grid, flow, eps=1e-12, validate=False):
        self.grid = grid
        self.flow = flow
        self.eps = float(eps)
        self.validate = bool(validate)

    # ------------------------------------------------------------------
    # Common helpers
    # ------------------------------------------------------------------

    def _check_scalar(self, p, w):
        """Return True when p has shape (d,) and w is scalar."""
        p = cp.asarray(p)
        w = cp.asarray(w)
        return p.ndim == 1 and w.ndim == 0

    def _as_vector(self, p):
        return cp.asarray(p, dtype=cp.float64)

    def _momentum_data(self, p):
        """Return p, |p|^2 and |p| for scalar or batched momentum."""
        p = self._as_vector(p)

        if p.ndim < 1:
            raise ValueError("p must have shape (d,) or (..., d).")

        p2 = cp.sum(p * p, axis=-1)
        pmag = cp.sqrt(p2)

        return p, p2, pmag

    def _safe_frequency_ratio(self, value, omega):
        """GPU batch-compatible evaluation of value / omega."""
        value, omega = cp.broadcast_arrays(cp.asarray(value), cp.asarray(omega, dtype=cp.float64))

        result_dtype = cp.result_type(value.dtype, cp.float64)
        result = cp.empty(value.shape, dtype=result_dtype)

        regular = cp.abs(omega) >= self.eps
        cp.divide(value, omega, out=result, where=regular)

        small_result = cp.where(cp.abs(value) < self.eps, 0.0, value / self.eps)
        result = cp.where(regular, result, small_result)

        return result.reshape(()) if result.ndim == 0 else result

    def _safe_frequency_square_ratio(self, value, omega):
        """GPU batch-compatible evaluation of value / omega**2."""
        value, omega = cp.broadcast_arrays(cp.asarray(value), cp.asarray(omega, dtype=cp.float64))

        result_dtype = cp.result_type(value.dtype, cp.float64)
        result = cp.empty(value.shape, dtype=result_dtype)

        regular = cp.abs(omega) >= self.eps
        cp.divide(value, omega * omega, out=result, where=regular)

        small_result = cp.where(cp.abs(value) < self.eps, 0.0, value / self.eps**2)
        result = cp.where(regular, result, small_result)

        return result.reshape(()) if result.ndim == 0 else result

    @staticmethod
    def _restore_scalar_output(value, scalar_output):
        """Keep scalar results as zero-dimensional CuPy arrays."""
        value = cp.asarray(value)
        return value.reshape(()) if scalar_output else value

    def _validate_nonzero(self, values, eps, message):
        """
        Optional validation.

        This synchronizes the GPU and should normally remain disabled
        inside the hot integration loop.
        """
        if self.validate and bool(cp.any(cp.abs(values) < eps).item()):
            bad_indices = cp.asnumpy(cp.flatnonzero(cp.abs(values) < eps)).tolist()
            raise ZeroDivisionError(f"{message} Invalid batch indices: {bad_indices}")

    # ------------------------------------------------------------------
    # Propagators
    # ------------------------------------------------------------------

    def G_uu_and_uubar(self, p, omega, k_rg, eps=1e-14):
        """
        GPU-compatible propagators.

        Supported momentum shapes:
            (d,)
            (B, d)
            (..., d)

        Frequencies must be broadcast-compatible with p.shape[:-1].
        """
        scalar_output = self._check_scalar(p, omega)

        p_batch = cp.asarray(p, dtype=cp.float64)
        omega_batch = cp.asarray(omega, dtype=cp.float64)

        if p_batch.ndim < 1:
            raise ValueError("p must have shape (d,) or (..., d).")

        p_sq = cp.sum(p_batch * p_batch, axis=-1)

        self._validate_nonzero(p_sq, eps, "The transverse propagator is undefined at |p|=0.")

        p_mag = cp.sqrt(p_sq)

        fnu_plus = self.grid.f_nu(p_mag, omega_batch)
        fnu_minus = self.grid.f_nu(p_mag, -omega_batch)
        fD = self.grid.f_d(p_mag, omega_batch)

        R = self.flow.R_scalar(p_batch, k_rg)
        Nkappa = self.flow.Nkappa_scalar(p_batch, k_rg)

        tau11_plus = -p_sq * fnu_plus - 1j * omega_batch + R
        tau11_minus = -p_sq * fnu_minus + 1j * omega_batch + R
        tau02 = p_sq**2 * fD - 2.0 * Nkappa

        denominator_uu = tau11_plus * tau11_minus

        self._validate_nonzero(denominator_uu, eps, "G_uu denominator is zero or numerically too small.")
        self._validate_nonzero(tau11_minus, eps, "G_uubar denominator is zero or numerically too small.")

        G_uu = tau02 / denominator_uu
        G_uubar = 1.0 / tau11_minus

        return self._restore_scalar_output(G_uu, scalar_output), self._restore_scalar_output(G_uubar, scalar_output)

    # ------------------------------------------------------------------
    # Three-point frequency functions
    # ------------------------------------------------------------------

    def A_v(self, p, omega1, omega2):
        """GPU batch-compatible A_v."""
        scalar_output = self._check_scalar(p, omega1)

        p_batch = cp.asarray(p, dtype=cp.float64)
        omega1_batch = cp.asarray(omega1, dtype=cp.float64)
        omega2_batch = cp.asarray(omega2, dtype=cp.float64)

        p_mag = cp.linalg.norm(p_batch, axis=-1)
        omega12 = omega1_batch + omega2_batch

        term1 = self._safe_frequency_ratio(self.grid.f_nu(p_mag, omega12), omega12)
        term2 = self._safe_frequency_ratio(self.grid.f_nu(p_mag, omega1_batch), omega1_batch)

        return self._restore_scalar_output(term1 + term2, scalar_output)

    def A_d(self, p, omega1, omega2):
        """GPU batch-compatible A_d."""
        scalar_output = self._check_scalar(p, omega1)

        p_batch = cp.asarray(p, dtype=cp.float64)
        omega1_batch = cp.asarray(omega1, dtype=cp.float64)
        omega2_batch = cp.asarray(omega2, dtype=cp.float64)

        p_mag = cp.linalg.norm(p_batch, axis=-1)
        result = self.grid.f_d(p_mag, omega1_batch) + self.grid.f_d(p_mag, omega2_batch)

        return self._restore_scalar_output(result, scalar_output)

    # ------------------------------------------------------------------
    # Four-point frequency functions
    # ------------------------------------------------------------------

    def D_v(self, p, omega1, omega2, omega3):
        """GPU batch-compatible D_v."""
        scalar_output = self._check_scalar(p, omega1)

        p_batch = cp.asarray(p, dtype=cp.float64)
        omega1_batch = cp.asarray(omega1, dtype=cp.float64)
        omega2_batch = cp.asarray(omega2, dtype=cp.float64)
        omega3_batch = cp.asarray(omega3, dtype=cp.float64)

        p_mag = cp.linalg.norm(p_batch, axis=-1)
        omega12 = omega1_batch + omega2_batch
        omega123 = omega12 + omega3_batch

        term1 = self._safe_frequency_square_ratio(self.grid.f_nu(p_mag, omega123), omega123)
        term2 = self._safe_frequency_square_ratio(self.grid.f_nu(p_mag, omega12), omega12)
        term3 = self._safe_frequency_square_ratio(self.grid.f_nu(p_mag, omega1_batch), omega1_batch)

        return self._restore_scalar_output(term1 + term2 + term3, scalar_output)

    def D_d(self, p, omega1, omega2, omega3):
        """GPU batch-compatible D_d."""
        scalar_output = self._check_scalar(p, omega1)

        p_batch = cp.asarray(p, dtype=cp.float64)
        omega1_batch = cp.asarray(omega1, dtype=cp.float64)
        omega2_batch = cp.asarray(omega2, dtype=cp.float64)
        omega3_batch = cp.asarray(omega3, dtype=cp.float64)

        p_mag = cp.linalg.norm(p_batch, axis=-1)
        omega12 = omega1_batch + omega2_batch
        omega123 = omega12 + omega3_batch

        term1 = self._safe_frequency_square_ratio(self.grid.f_d(p_mag, omega123), omega123)
        term2 = self._safe_frequency_square_ratio(self.grid.f_d(p_mag, omega12), omega12)
        term3 = self._safe_frequency_square_ratio(self.grid.f_d(p_mag, omega1_batch), omega1_batch)

        return self._restore_scalar_output(term1 + term2 + term3, scalar_output)


########### Important...do flush() after every updation ################
class WetterichSolver:
    """
    Hybrid CPU/GPU Wetterich solver.

    CPU / NumPy:
        - coordinate and RG grids
        - stored f_nu and f_d stacks
        - sequential RG-scale loop
        - Euler/overwrite updates

    GPU / CuPy:
        - external/internal Cartesian-product batching
        - projectors, propagators and coefficient contractions
        - direct integrands and quadrature reduction

    FRGGrid, FlowParameters and Vertices are expected to be the CuPy-compatible
    versions used by this solver.
    """

    def __init__(self, flow, p_grid, omega_grid, k_grid, f_nu_stack, f_d_stack, *, q_cutoff_factor=10.0, flow_variable="s", update_mode="euler", propagator_output="scalar", eps=1e-14, validate_gpu=False):
        self.flow = flow
        self.p_grid = np.asarray(p_grid, dtype=float)
        self.omega_grid = np.asarray(omega_grid, dtype=float)
        self.k_grid = np.asarray(k_grid, dtype=float)
        self.f_nu_stack = np.asarray(f_nu_stack, dtype=float)
        self.f_d_stack = np.asarray(f_d_stack, dtype=float)
        self.q_cutoff_factor = float(q_cutoff_factor)
        self.flow_variable = flow_variable
        self.update_mode = update_mode
        self.propagator_output = propagator_output
        self.eps = float(eps)
        self.validate_gpu = bool(validate_gpu)

        if flow_variable not in {"s", "k"}:
            raise ValueError("flow_variable must be 's' or 'k'.")
        if update_mode not in {"overwrite", "euler"}:
            raise ValueError("update_mode must be 'overwrite' or 'euler'.")
        if propagator_output != "scalar":
            raise ValueError("The CuPy solver requires propagator_output='scalar'.")
        if self.p_grid.ndim != 1 or self.omega_grid.ndim != 1 or self.k_grid.ndim != 1:
            raise ValueError("p_grid, omega_grid and k_grid must be one-dimensional.")
        if self.k_grid.size < 2:
            raise ValueError("k_grid must contain at least two RG scales.")
        expected_stack_shape = (self.k_grid.size, self.p_grid.size, self.omega_grid.size)
        if self.f_nu_stack.shape != expected_stack_shape:
            raise ValueError(f"f_nu_stack has the wrong shape. Expected {expected_stack_shape}, received {self.f_nu_stack.shape}.")
        if self.f_d_stack.shape != expected_stack_shape:
            raise ValueError(f"f_d_stack has the wrong shape. Expected {expected_stack_shape}, received {self.f_d_stack.shape}.")
        if not np.all(np.diff(self.p_grid) > 0.0):
            raise ValueError("p_grid must be strictly increasing.")
        if not np.all(np.diff(self.omega_grid) > 0.0):
            raise ValueError("omega_grid must be strictly increasing.")
        if np.any(self.k_grid <= 0.0):
            raise ValueError("Every RG scale in k_grid must be positive.")

    # ==================================================================
    # CPU setup helpers
    # ==================================================================

    @staticmethod
    def _delta(i, j):
        return 1.0 if i == j else 0.0

    @staticmethod
    def p_external_vector(p_scalar):
        """Choose the external momentum along the z direction."""
        return np.array([0.0, 0.0, float(p_scalar)], dtype=float)

    @staticmethod
    def _trapezoid_weights(n):
        weights = np.ones(n, dtype=float)
        if n > 1:
            weights[0] = 0.5
            weights[-1] = 0.5
        return weights

    def internal_grids(self, k_rg, Nq_int, Nomega_int):
        """Construct the symmetric CPU quadrature grids for one RG scale."""
        if Nq_int < 2 or Nomega_int < 2:
            raise ValueError("Nq_int and Nomega_int must both be at least 2.")
        limit = self.q_cutoff_factor * float(k_rg)
        q_grid = np.linspace(-limit, limit, Nq_int)
        Omega_grid = np.linspace(-limit, limit, Nomega_int)
        dq = q_grid[1] - q_grid[0]
        dOmega = Omega_grid[1] - Omega_grid[0]
        return q_grid, Omega_grid, dq, dOmega, self._trapezoid_weights(Nq_int), self._trapezoid_weights(Nomega_int)

    def make_grid_object(self, k_index):
        """Build the CuPy FRGGrid for the current CPU-stored RG slice."""
        return FRGGrid(self.p_grid, self.omega_grid, self.f_nu_stack[k_index], self.f_d_stack[k_index])

    # ==================================================================
    # GPU helpers
    # ==================================================================

    def _projector_matrix_batch(self, momenta):
        """Construct P_ij(p)=delta_ij-p_i p_j/p^2 for momenta shaped (...,d)."""
        momenta = cp.asarray(momenta, dtype=cp.float64)
        if momenta.ndim < 1:
            raise ValueError("momenta must have shape (d,) or (..., d).")
        momentum_sq = cp.sum(momenta * momenta, axis=-1)
        safe_momentum_sq = cp.where(momentum_sq >= self.eps, momentum_sq, 1.0)
        identity = cp.eye(momenta.shape[-1], dtype=cp.float64)
        return identity - momenta[..., :, None] * momenta[..., None, :] / safe_momentum_sq[..., None, None]

    def _dR(self, q, k_rg):
        result = self.flow.dRds_scalar(q, k_rg)
        return result / k_rg if self.flow_variable == "k" else result

    def _validate_nonzero_gpu(self, values, message):
        """Optional synchronized validation; keep disabled in production."""
        if self.validate_gpu and bool(cp.any(cp.abs(values) < self.eps).item()):
            bad_indices = cp.asnumpy(cp.flatnonzero(cp.abs(values) < self.eps)).tolist()
            raise ZeroDivisionError(f"{message} Invalid flattened indices: {bad_indices}")

    # ==================================================================
    # Equations (8) and (9): four-point coefficients for (1,1)
    # ==================================================================

    def _coefficients_nu_four_point(self, vertices, p, q, w1, w2):
        p = cp.asarray(p, dtype=cp.float64)
        q = cp.asarray(q, dtype=cp.float64)
        w1 = cp.asarray(w1, dtype=cp.float64)
        w2 = cp.asarray(w2, dtype=cp.float64)
        batch_size, d_space = q.shape
        p2 = cp.einsum("bi,bi->b", p, p)
        q2 = cp.einsum("bi,bi->b", q, q)
        p_plus_q = p + q
        p_minus_q = p - q
        q_minus_p = q - p
        P_p = self._projector_matrix_batch(p)
        P_q = self._projector_matrix_batch(q)
        Dv_1 = vertices.D_v(q, w1, w2, -w2)
        Dv_2 = vertices.D_v(q, w1, -w2, w2)
        Dv_3 = vertices.D_v(q, -w2, w1, w2)
        Dv_4 = vertices.D_v(q, w2, w1, -w2)
        Dd_1 = vertices.D_d(q, w2, w1, -w2)
        Dd_2 = vertices.D_d(p, w1, w2, -w1)
        c1 = cp.zeros(batch_size, dtype=cp.complex128)
        c2 = cp.zeros(batch_size, dtype=cp.complex128)

        for i in range(d_space):
            for j in range(d_space):
                Pij_p = P_p[:, i, j]
                Pij_q = P_q[:, i, j]
                for b in range(d_space):
                    for c in range(d_space):
                        Pbc_q = P_q[:, b, c]
                        for d_idx in range(d_space):
                            Pdb_q = P_q[:, d_idx, b]
                            brace_c1 = p[:, c] * p_plus_q[:, d_idx] * Pij_q * Dv_1 + p[:, d_idx] * p_minus_q[:, c] * Pij_q * Dv_2 - q[:, i] * p_minus_q[:, c] * P_q[:, j, d_idx] * Dv_3 + q[:, i] * p_plus_q[:, d_idx] * P_q[:, c, j] * Dv_4
                            brace_c2 = -q[:, i] * p_minus_q[:, c] * Dd_1 - p[:, c] * q_minus_p[:, i] * Dd_2
                            common = Pij_p * Pbc_q * Pdb_q
                            c1 += p2 * common * brace_c1
                            c2 += q2 * p2 * common * brace_c2

        return c1, c2

    # ==================================================================
    # Equations (10)-(14): three-point coefficients for (1,1)
    # ==================================================================

    def _coefficients_nu_three_point(self, vertices, p, q, w1, w2):
        p = cp.asarray(p, dtype=cp.float64)
        q = cp.asarray(q, dtype=cp.float64)
        w1 = cp.asarray(w1, dtype=cp.float64)
        w2 = cp.asarray(w2, dtype=cp.float64)
        batch_size, d_space = q.shape
        p2 = cp.einsum("bi,bi->b", p, p)
        q2 = cp.einsum("bi,bi->b", q, q)
        p_plus_q = p + q
        pq2 = cp.einsum("bi,bi->b", p_plus_q, p_plus_q)
        P_p = self._projector_matrix_batch(p)
        P_q = self._projector_matrix_batch(q)
        P_pq = self._projector_matrix_batch(p_plus_q)
        delta = cp.eye(d_space, dtype=cp.float64)
        Av_p = vertices.A_v(p, w1, w2)
        Av_q = vertices.A_v(q, w2, w1)
        Av_minus_q = vertices.A_v(-q, -w2, w1 + w2)
        Ad_minus_p = vertices.A_d(-p, -w1, w1 + w2)
        Ad_minus_q = vertices.A_d(-q, w1 + w2, -w2)
        Ad_pq = vertices.A_d(p_plus_q, w1 + w2, -w2)
        c1 = cp.zeros(batch_size, dtype=cp.complex128)
        c2 = cp.zeros(batch_size, dtype=cp.complex128)
        c3 = cp.zeros(batch_size, dtype=cp.complex128)

        for i in range(d_space):
            for j in range(d_space):
                Pij_p = P_p[:, i, j]
                for b in range(d_space):
                    for c in range(d_space):
                        Pbc_q = P_q[:, b, c]
                        common_external = Pij_p * Pbc_q
                        for d_idx in range(d_space):
                            left = -1j * (q[:, i] * delta[c, d_idx] + p[:, c] * delta[i, d_idx]) - pq2 * (-p[:, c] * P_p[:, i, d_idx] * Av_p - q[:, i] * P_q[:, c, d_idx] * Av_q)
                            c3_left = pq2 * q2 * (q[:, i] * P_q[:, c, d_idx] * Ad_minus_q - p_plus_q[:, i] * P_pq[:, c, d_idx] * Ad_pq)
                            for e in range(d_space):
                                Pde_pq = P_pq[:, d_idx, e]
                                for f in range(d_space):
                                    Pfb_q = P_q[:, f, b]
                                    c_double_prime = Pde_pq * (1j * q[:, j] * delta[e, f] - p2 * q[:, j] * P_q[:, e, f] * Av_minus_q) * Pfb_q
                                    c1 += common_external * left * c_double_prime
                                    c2 += common_external * left * Pde_pq * pq2 * p2 * p[:, e] * P_p[:, j, f] * Ad_minus_p * Pfb_q
                                    c3 += common_external * c3_left * c_double_prime

        return c1, c2, c3

    # ==================================================================
    # Equation (17): four-point coefficient for (2,0)
    # ==================================================================

    def _coefficient_d_four_point(self, vertices, p, q, w1, w2):
        p = cp.asarray(p, dtype=cp.float64)
        q = cp.asarray(q, dtype=cp.float64)
        w1 = cp.asarray(w1, dtype=cp.float64)
        w2 = cp.asarray(w2, dtype=cp.float64)
        batch_size, d_space = q.shape
        q2 = cp.einsum("bi,bi->b", q, q)
        p_plus_q = p + q
        q_minus_p = q - p
        P_p = self._projector_matrix_batch(p)
        P_q = self._projector_matrix_batch(q)
        D1 = vertices.D_d(-p, w1, w2, -w1)
        D2 = vertices.D_d(p, w2, w1, -w1)
        D3 = vertices.D_d(p, -w1, w2, w1)
        D4 = vertices.D_d(p, -w2, -w1, w1)
        coefficient = cp.zeros(batch_size, dtype=cp.complex128)

        for i in range(d_space):
            for j in range(d_space):
                Pij_p = P_p[:, i, j]
                for b in range(d_space):
                    for c in range(d_space):
                        Pbc_q = P_q[:, b, c]
                        for d_idx in range(d_space):
                            Pdb_q = P_q[:, d_idx, b]
                            brace = p[:, c] * p_plus_q[:, j] * P_p[:, i, d_idx] * D1 + q[:, i] * p_plus_q[:, j] * P_p[:, c, d_idx] * D2 - p[:, c] * q_minus_p[:, i] * P_p[:, j, d_idx] * D3 + q[:, j] * q_minus_p[:, i] * P_p[:, d_idx, c] * D4
                            coefficient += Pij_p * Pbc_q * q2 * brace * Pdb_q

        return coefficient

    # ==================================================================
    # Equations (18)-(20): three-point coefficients for (2,0)
    # ==================================================================

    def _coefficients_d_three_point(self, vertices, p, q, w1, w2):
        p = cp.asarray(p, dtype=cp.float64)
        q = cp.asarray(q, dtype=cp.float64)
        w1 = cp.asarray(w1, dtype=cp.float64)
        w2 = cp.asarray(w2, dtype=cp.float64)
        batch_size, d_space = q.shape
        q2 = cp.einsum("bi,bi->b", q, q)
        p_plus_q = p + q
        pq2 = cp.einsum("bi,bi->b", p_plus_q, p_plus_q)
        P_p = self._projector_matrix_batch(p)
        P_q = self._projector_matrix_batch(q)
        P_pq = self._projector_matrix_batch(p_plus_q)
        delta = cp.eye(d_space, dtype=cp.float64)
        Av_p = vertices.A_v(p, w1, w2)
        Av_q = vertices.A_v(q, w2, w1)
        Av_minus_p = vertices.A_v(-p, -w1, w1 + w2)
        Av_pq = vertices.A_v(p_plus_q, w1 + w2, -w1)
        Ad_minus_q = vertices.A_d(-q, w1 + w2, -w2)
        Ad_pq = vertices.A_d(p_plus_q, w1 + w2, -w2)
        c1 = cp.zeros(batch_size, dtype=cp.complex128)
        c2 = cp.zeros(batch_size, dtype=cp.complex128)

        for i in range(d_space):
            for j in range(d_space):
                Pij_p = P_p[:, i, j]
                for b in range(d_space):
                    for c in range(d_space):
                        Pbc_q = P_q[:, b, c]
                        common_external = Pij_p * Pbc_q
                        for d_idx in range(d_space):
                            left = -1j * (q[:, i] * delta[c, d_idx] + p[:, c] * delta[i, d_idx]) - pq2 * (-p[:, c] * P_p[:, i, d_idx] * Av_p) - q[:, i] * P_q[:, c, d_idx] * Av_q
                            for e in range(d_space):
                                Pde_pq = P_pq[:, d_idx, e]
                                c_triple_prime = common_external * left * Pde_pq
                                for f in range(d_space):
                                    Pfb_q = P_q[:, f, b]
                                    remainder_c1 = -1j * (p_plus_q[:, j] * delta[e, f] - p[:, e] * delta[f, j]) - q2 * (p[:, e] * P_p[:, f, j] * Av_minus_p - p_plus_q[:, j] * P_pq[:, e, f] * Av_pq)
                                    remainder_c2 = q2 * pq2 * (q[:, j] * P_q[:, e, f] * Ad_minus_q - p_plus_q[:, j] * P_pq[:, e, f] * Ad_pq)
                                    c1 += c_triple_prime * remainder_c1 * Pfb_q
                                    c2 += c_triple_prime * remainder_c2 * Pfb_q

        return c1, c2

    # ==================================================================
    # Direct flattened integrands
    # ==================================================================

    def direct_integrands(self, vertices, p, omega, q, Omega, k_rg):
        """Evaluate one flattened external/internal pair batch entirely on GPU."""
        p = cp.asarray(p, dtype=cp.float64)
        omega = cp.asarray(omega, dtype=cp.float64)
        q = cp.asarray(q, dtype=cp.float64)
        Omega = cp.asarray(Omega, dtype=cp.float64)

        if p.ndim != 2 or q.ndim != 2:
            raise ValueError("p and q must have shape (E*B, d).")
        if omega.ndim != 1 or Omega.ndim != 1:
            raise ValueError("omega and Omega must have shape (E*B,).")
        if p.shape != q.shape:
            raise ValueError("p and q must have identical shapes.")
        if not (p.shape[0] == omega.size == Omega.size):
            raise ValueError("p, q, omega and Omega must have the same flattened batch size.")

        batch_size, d_space = q.shape
        p_plus_q = p + q
        q2 = cp.einsum("bi,bi->b", q, q)
        pq2 = cp.einsum("bi,bi->b", p_plus_q, p_plus_q)
        valid = (q2 >= self.eps) & (pq2 >= self.eps)

        # Replace only masked points by a safe dummy momentum. Their final
        # contributions are set to zero, but no dynamic boolean compaction or
        # zero-momentum projector/propagator evaluation is required.
        safe_q_vector = cp.zeros(d_space, dtype=cp.float64)
        safe_q_vector[0] = max(1.0, float(k_rg))
        q_safe = cp.where(valid[:, None], q, safe_q_vector[None, :]) # if q is close to 0, it is instead taken as safe_q_vector
        p_plus_q_safe = p + q_safe
        shifted_frequency = omega + Omega

        Guu_q, Guubar_q = vertices.G_uu_and_uubar(q_safe, Omega, k_rg)
        _, Guubar_q_bar = vertices.G_uu_and_uubar(q_safe, -Omega, k_rg)
        Guu_pq_prime, Guubar_pq_prime = vertices.G_uu_and_uubar(p_plus_q_safe, shifted_frequency, k_rg)
        _, Guubar_pq_bar_prime = vertices.G_uu_and_uubar(p_plus_q_safe, -shifted_frequency, k_rg)
        dR = self._dR(q_safe, k_rg)

        c1_4_nu, c2_4_nu = self._coefficients_nu_four_point(vertices, p, q_safe, omega, Omega)
        c1_3_nu, c2_3_nu, c3_3_nu = self._coefficients_nu_three_point(vertices, p, q_safe, omega, Omega)
        trace4_nu = dR * (c1_4_nu * (Guubar_q_bar * Guu_q + Guu_q * Guubar_q) + c2_4_nu * (Guubar_q_bar**2 + Guubar_q**2))
        trace3_nu = dR * (c1_3_nu * (Guubar_pq_bar_prime * (Guu_q * Guubar_q_bar + Guubar_q * Guu_q) + Guu_pq_prime * Guubar_q**2) + c2_3_nu * Guubar_pq_prime * (Guubar_q_bar**2 + Guubar_q**2) + c3_3_nu * Guubar_pq_prime * Guubar_q**2)
        rhs_nu = cp.where(valid, -0.5 * trace4_nu + trace3_nu, 0.0)

        c4_d = self._coefficient_d_four_point(vertices, p, q_safe, omega, Omega)
        c1_3_d, c2_3_d = self._coefficients_d_three_point(vertices, p, q_safe, omega, Omega)
        trace4_d = dR * c4_d * (Guubar_q_bar**2 + Guubar_q**2)
        trace3_d = dR * (c1_3_d * Guubar_q_bar**2 + c2_3_d * Guubar_q**2)
        rhs_d = cp.where(valid, -0.5 * trace4_d + trace3_d, 0.0)

        return rhs_nu, rhs_d

    # ==================================================================
    # Flattened external/internal GPU quadrature
    # ==================================================================

    def projected_rhs_flattened_gpu(self, vertices, k_rg, Nq_int, Nomega_int, external_batch_size=8, internal_batch_size=1024, show_progress=False):
        if external_batch_size < 1 or internal_batch_size < 1:
            raise ValueError("external_batch_size and internal_batch_size must be at least 1.")

        p_grid_gpu = cp.asarray(self.p_grid, dtype=cp.float64)
        omega_grid_gpu = cp.asarray(self.omega_grid, dtype=cp.float64)
        p_scalar_mesh, omega_mesh = cp.meshgrid(p_grid_gpu, omega_grid_gpu, indexing="ij")
        p_scalar_external = p_scalar_mesh.ravel()
        omega_external = omega_mesh.ravel()
        number_external = p_scalar_external.size
        external_direction = cp.asarray(self.p_external_vector(1.0), dtype=cp.float64)
        p_external = p_scalar_external[:, None] * external_direction[None, :]
        valid_external = cp.abs(p_scalar_external) > self.eps
        p_external_safe = cp.where(valid_external[:, None], p_external, external_direction[None, :])   # isf it is invalid, P is instead chosen as 0,0,1

        q_grid, Omega_grid, dq, dOmega, q_weights, Omega_weights = self.internal_grids(k_rg, Nq_int, Nomega_int)
        q_grid = cp.asarray(q_grid, dtype=cp.float64)
        Omega_grid = cp.asarray(Omega_grid, dtype=cp.float64)
        q_weights = cp.asarray(q_weights, dtype=cp.float64)
        Omega_weights = cp.asarray(Omega_weights, dtype=cp.float64)
        QX, QY, QZ = cp.meshgrid(q_grid, q_grid, q_grid, indexing="ij")
        q_spatial = cp.column_stack((QX.ravel(), QY.ravel(), QZ.ravel()))
        spatial_weights = (q_weights[:, None, None] * q_weights[None, :, None] * q_weights[None, None, :]).ravel()
        number_spatial = q_spatial.shape[0]
        n_omega = Omega_grid.size
        q_points = cp.repeat(q_spatial, n_omega, axis=0)
        Omega_points = cp.tile(Omega_grid, number_spatial)
        integration_weights = cp.repeat(spatial_weights, n_omega) * cp.tile(Omega_weights, number_spatial)
        integration_weights *= float(dq)**3 * float(dOmega) / (2.0 * np.pi)**4
        number_internal = q_points.shape[0]

        rhs_nu_external = cp.zeros(number_external, dtype=cp.complex128)
        rhs_d_external = cp.zeros(number_external, dtype=cp.complex128)
        projector_norm = 2.0

        for external_start in range(0, number_external, external_batch_size):
            external_stop = min(external_start + external_batch_size, number_external)
            p_chunk = p_external_safe[external_start:external_stop]
            omega_chunk = omega_external[external_start:external_stop]
            valid_chunk = valid_external[external_start:external_stop]
            current_external_size = p_chunk.shape[0]
            integral_nu = cp.zeros(current_external_size, dtype=cp.complex128)
            integral_d = cp.zeros(current_external_size, dtype=cp.complex128)

            for internal_start in range(0, number_internal, internal_batch_size):
                internal_stop = min(internal_start + internal_batch_size, number_internal)
                q_chunk = q_points[internal_start:internal_stop]
                Omega_chunk = Omega_points[internal_start:internal_stop]
                weight_chunk = integration_weights[internal_start:internal_stop]
                current_internal_size = q_chunk.shape[0]
                p_pairs = cp.repeat(p_chunk, current_internal_size, axis=0)
                omega_pairs = cp.repeat(omega_chunk, current_internal_size)
                q_pairs = cp.tile(q_chunk, (current_external_size, 1))
                Omega_pairs = cp.tile(Omega_chunk, current_external_size)
                integrand_nu_pairs, integrand_d_pairs = self.direct_integrands(vertices, p_pairs, omega_pairs, q_pairs, Omega_pairs, k_rg)
                integrand_nu = integrand_nu_pairs.reshape(current_external_size, current_internal_size)
                integrand_d = integrand_d_pairs.reshape(current_external_size, current_internal_size)
                integral_nu += cp.sum(integrand_nu * weight_chunk[None, :], axis=1)
                integral_d += cp.sum(integrand_d * weight_chunk[None, :], axis=1)

            rhs_nu_external[external_start:external_stop] = cp.where(valid_chunk, integral_nu / projector_norm, 0.0)   # if it is a valid point, it is taken as usual, otherwise trivially 0
            rhs_d_external[external_start:external_stop] = cp.where(valid_chunk, integral_d / projector_norm, 0.0)

            if show_progress:
                cp.cuda.get_current_stream().synchronize()
                percentage = 100.0 * external_stop / number_external
                print(f"k={k_rg:.6g}: external points {external_stop}/{number_external} ({percentage:.1f}%)")

        return rhs_nu_external.reshape(self.p_grid.size, self.omega_grid.size), rhs_d_external.reshape(self.p_grid.size, self.omega_grid.size)

    # ==================================================================
    # CPU storage/update and sequential solve
    # ==================================================================

    def _store_next_grid(self, nk, rhs_nu_gpu, rhs_d_gpu):
        """Copy one completed RHS grid to CPU and update the next RG slice."""
        rhs_nu = cp.asnumpy(rhs_nu_gpu)
        rhs_d = cp.asnumpy(rhs_d_gpu)
        p2 = self.p_grid[:, None]**2
        p4 = self.p_grid[:, None]**4
        valid_p = np.abs(self.p_grid) > self.eps
        flow_nu = np.zeros(rhs_nu.shape, dtype=float)
        flow_d = np.zeros(rhs_d.shape, dtype=float)
        flow_nu[valid_p, :] = np.real(rhs_nu[valid_p, :] / (3.0 * p2[valid_p, :]))  # at invalid p, it is trivially 0
        flow_d[valid_p, :] = np.real(rhs_d[valid_p, :] / (3.0 * p4[valid_p, :]))

        if self.update_mode == "overwrite":
            self.f_nu_stack[nk + 1] = flow_nu
            self.f_d_stack[nk + 1] = flow_d
            return

        step = np.log(self.k_grid[nk + 1] / self.k_grid[nk]) if self.flow_variable == "s" else self.k_grid[nk + 1] - self.k_grid[nk]
        self.f_nu_stack[nk + 1] = self.f_nu_stack[nk] + step * flow_nu
        self.f_d_stack[nk + 1] = self.f_d_stack[nk] + step * flow_d

    def solve(self, Nq_int, Nomega_int, *, external_batch_size=8, internal_batch_size=1024, show_progress=False):
        """Solve sequentially in k while evaluating each complete RHS grid on GPU."""
        number_steps = self.k_grid.size - 1

        for nk in range(number_steps):
            k_rg = float(self.k_grid[nk])
            grid = self.make_grid_object(nk)
            vertices = Vertices(grid, self.flow, eps=self.eps, validate=self.validate_gpu)
            rhs_nu_gpu, rhs_d_gpu = self.projected_rhs_flattened_gpu(vertices, k_rg, Nq_int, Nomega_int, external_batch_size=external_batch_size, internal_batch_size=internal_batch_size, show_progress=False)
            self._store_next_grid(nk, rhs_nu_gpu, rhs_d_gpu)

            if show_progress:
                percentage = 100.0 * (nk + 1) / number_steps
                print(f"RG step {nk + 1}/{number_steps}, k={k_rg:.6g}, progress={percentage:.1f}%")

        return self.f_nu_stack, self.f_d_stack
                
                      
                

In [3]:
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# Grid sizes
# ------------------------------------------------------------

Nk = 10
Np = 10
Nw = 10

# ------------------------------------------------------------
# Grids
# ------------------------------------------------------------

d = 3

p_min = 0.1
p_max = 300

omega_min = 0.1
omega_max = 300

k_start = 1
k_end = 1e-3

# ------------------------------------------------------------
# Log-spaced p grid
# ------------------------------------------------------------

#p_grid = np.geomspace(p_min, p_max, Np)

# ------------------------------------------------------------
# Symmetric log-spaced omega grid
# ------------------------------------------------------------

Nw_half = Nw // 2

p_pos = np.geomspace(omega_min, omega_max, Nw_half)
p_neg = -p_pos[::-1]
p_grid = np.concatenate([p_neg,p_pos])

omega_positive = np.geomspace(omega_min, omega_max, Nw_half)
omega_negative = -omega_positive[::-1]

omega_grid = np.concatenate([omega_negative, omega_positive])

# ------------------------------------------------------------
# Log-spaced k grid from 1 to 1e-6
# ------------------------------------------------------------

k_grid = np.geomspace(k_start, k_end, Nk)
# ------------------------------------------------------------
# Create output folder
# ------------------------------------------------------------

data_dir = Path.cwd() / "frg_data"
data_dir.mkdir(exist_ok=True)

f_v_path = data_dir / "f_v_stack.npy"
f_d_path = data_dir / "f_d_stack.npy"

f_v_stack = np.lib.format.open_memmap(
    f_v_path,
    dtype="float64",
    mode="w+",
    shape=(Nk, Np, Nw),
)

f_d_stack = np.lib.format.open_memmap(
    f_d_path,
    dtype="float64",
    mode="w+",
    shape=(Nk, Np, Nw),
)

f_v_stack[:] = 0.1
f_d_stack[:] = 0.1

f_v_stack.flush()
f_d_stack.flush()

In [4]:
print(p_grid)

[-3.00000000e+02 -4.05360046e+01 -5.47722558e+00 -7.40082804e-01
 -1.00000000e-01  1.00000000e-01  7.40082804e-01  5.47722558e+00
  4.05360046e+01  3.00000000e+02]


In [5]:
one_stack_memory_GB = Nk * Np * Nw * 8 / 1e9
two_stack_memory_GB = 2 * one_stack_memory_GB

print(one_stack_memory_GB, "GB per stack")
print(two_stack_memory_GB, "GB total")

8e-06 GB per stack
1.6e-05 GB total


In [69]:


# ------------------------------------------------------------
# Define the RG-flow parameters
# ------------------------------------------------------------

flow = FlowParameters(
    nu0=1.0,
    D0=1.0,
    k0=k_start,
    eta_nu=4.0 / 3.0,
    eta_D=3.0,
    a=0.5,
)

# ------------------------------------------------------------
# Construct the Wetterich solver
# ------------------------------------------------------------

solver = WetterichSolver(
    flow=flow,
    p_grid=p_grid,
    omega_grid=omega_grid,
    k_grid=k_grid,
    f_nu_stack=f_v_stack,
    f_d_stack=f_d_stack
)

# ------------------------------------------------------------
# Internal integration-grid sizes
# ------------------------------------------------------------
# Start with small values because the number of integration points is:
#
#     Nq_int^3 * Nomega_int
#
# Nq_int = 3 and Nomega_int = 3 gives:
#
#     3^3 * 3 = 81
#
# internal points for every external (k, p, omega) point.

Nq_int = 50
Nomega_int = 50

# ------------------------------------------------------------
# Solve the flow equations
# ------------------------------------------------------------

start_time = time.perf_counter()

try:
    f_v_stack, f_d_stack = solver.solve(
        Nq_int=Nq_int,
        Nomega_int=Nomega_int,
        show_progress=True,
    )

finally:
    # Flush results even if execution is manually stopped or an
    # error occurs after some RG steps have been completed.
    if hasattr(f_v_stack, "flush"):
        f_v_stack.flush()

    if hasattr(f_d_stack, "flush"):
        f_d_stack.flush()

elapsed_time = time.perf_counter() - start_time

print("\n" + "=" * 70)
print("Wetterich flow calculation completed")
print(f"Total execution time: {elapsed_time:.6f} seconds")
print(f"Total execution time: {elapsed_time / 60.0:.3f} minutes")
print(f"Total execution time: {elapsed_time / 3600.0:.3f} hours")
print("=" * 70)


Starting RG step: k_index=0/8, k=1


C:\Users\Siddharth\AppData\Local\Temp\ipykernel_19864\3629130275.py:454: RuntimeWarning: overflow encountered in expm1
  denominator = np.expm1(x)


k=1, p=-300, omega=-300: 0.0%
k=1, p=-300, omega=-300: 0.1%
k=1, p=-300, omega=-300: 0.1%
k=1, p=-300, omega=-300: 0.1%
k=1, p=-300, omega=-300: 0.2%
k=1, p=-300, omega=-300: 0.2%
k=1, p=-300, omega=-300: 0.2%
k=1, p=-300, omega=-300: 0.3%
k=1, p=-300, omega=-300: 0.3%
k=1, p=-300, omega=-300: 0.3%
k=1, p=-300, omega=-300: 0.4%
k=1, p=-300, omega=-300: 0.4%
k=1, p=-300, omega=-300: 0.4%
k=1, p=-300, omega=-300: 0.4%
k=1, p=-300, omega=-300: 0.5%
k=1, p=-300, omega=-300: 0.5%
k=1, p=-300, omega=-300: 0.5%
k=1, p=-300, omega=-300: 0.6%
k=1, p=-300, omega=-300: 0.6%
k=1, p=-300, omega=-300: 0.6%
k=1, p=-300, omega=-300: 0.7%
k=1, p=-300, omega=-300: 0.7%
k=1, p=-300, omega=-300: 0.7%
k=1, p=-300, omega=-300: 0.8%
k=1, p=-300, omega=-300: 0.8%
k=1, p=-300, omega=-300: 0.8%
k=1, p=-300, omega=-300: 0.9%
k=1, p=-300, omega=-300: 0.9%
k=1, p=-300, omega=-300: 0.9%
k=1, p=-300, omega=-300: 1.0%
k=1, p=-300, omega=-300: 1.0%
k=1, p=-300, omega=-300: 1.0%
k=1, p=-300, omega=-300: 1.1%
k=1, p=-30

KeyboardInterrupt: 

In [ ]:
# ------------------------------------------------------------
# Save the coordinate grids
# ------------------------------------------------------------

np.save(data_dir / "p_grid.npy", p_grid)
np.save(data_dir / "omega_grid.npy", omega_grid)
np.save(data_dir / "k_grid.npy", k_grid)

# ------------------------------------------------------------
# Ensure all disk-backed results are written
# ------------------------------------------------------------

f_v_stack.flush()
f_d_stack.flush()

# ------------------------------------------------------------
# Save copies using the requested f_v and f_d notation
# ------------------------------------------------------------

np.save(
    data_dir / "f_v_stack.npy",
    np.asarray(f_v_stack),
)

np.save(
    data_dir / "f_d_stack.npy",
    np.asarray(f_d_stack),
)

# Save the final RG-scale slices separately
np.save(
    data_dir / "f_v_final.npy",
    np.asarray(f_v_stack[-1]),
)

np.save(
    data_dir / "f_d_final.npy",
    np.asarray(f_d_stack[-1]),
)

print("\nSaved files:")
print(data_dir / "f_v_stack.npy")
print(data_dir / "f_d_stack.npy")
print(data_dir / "f_v_final.npy")
print(data_dir / "f_d_final.npy")
print(data_dir / "p_grid.npy")
print(data_dir / "omega_grid.npy")
print(data_dir / "k_grid.npy")